In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [43]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=gemini_api_key)

In [51]:
query = "KT 소액결제 사건"

In [45]:
response = llm.invoke(query)
response

AIMessage(content='2025년에 KT 소액결제 사건이 실제로 발생했는지 확인하기 위해 검색을 시도했지만, 현재까지는 그러한 사건에 대한 공식적인 정보나 보도는 찾을 수 없었습니다. \n\n따라서 2025년 KT 소액결제 사건에 대한 구체적인 원인을 답변하는 것은 불가능합니다. \n\n만약 특정 사건에 대한 정보를 얻고 싶으시다면, 다음 사항을 확인해 보시는 것을 추천합니다.\n\n*   **정확한 날짜와 키워드:** "KT 소액결제", "2025년", "해킹", "개인정보 유출" 등 더 구체적인 키워드를 조합하여 검색해 보세요.\n*   **공식 발표 및 언론 보도:** KT 공식 홈페이지, 뉴스 매체, IT 관련 커뮤니티 등을 통해 공식 발표나 관련 보도가 있는지 확인해 보세요.\n*   **보안 관련 웹사이트:** 한국인터넷진흥원(KISA) 등 보안 관련 기관의 웹사이트에서 관련 정보를 찾아보세요.\n\n만약 미래에 실제로 2025년 KT 소액결제 사건이 발생한다면, 위와 같은 방법으로 원인과 관련된 정보를 얻으실 수 있을 것입니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--0ca3b486-aa47-45a8-a144-7ab807cd7af8-0', usage_metadata={'input_tokens': 21, 'output_tokens': 315, 'total_tokens': 336, 'input_token_details': {'cache_read': 0}})

In [52]:
from langchain_community.tools import DuckDuckGoSearchResults 

search = DuckDuckGoSearchResults(results_separator=';\n')
docs = search.invoke(query)

print(docs)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch07\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


snippet: 4 days ago · 서울 영등포구 에서도 유사한 사건이 발생했던 것으로 확인됐다. 서울 영등포경찰서는 지난달 (8월) 6일 오후 "소액결제로 49만 5천 원이 빠져나갔다"는 30대 KT 가입자의 신고를 접수했다. …, title: KT 소액결제 해킹 사건 - 나무위키, link: https://namu.wiki/w/KT+소액결제+해킹+사건;
snippet: 2 days ago · 수도권 특정 지역의 KT 이용자들이 휴대전화 소액결제 피해를 보는 사례가 잇따르는 가운데 KT가 이번 사건 초기 경찰로부터 발생 사실을 통보 ..., title: KT, 소액결제 사건 초기 경찰서 통보받아…'늑장 대처' 의혹, link: https://news.sbs.co.kr/news/endPage.do?news_id=N1008250962;
snippet: 2 days ago · 지난달 하순부터 확인된 KT의 소액결제 사태와 관련해 경찰이 그 전에 이상징후를 통보했음에도 이를 KT 측이 "이상없음"이라고 답했다는 지적이 제기됐다. 10일 국회 과학기술정보방송통신위원회 소속 …, title: KT, '소액결제 사건' 경찰 통보 받고도 무시…4일 뒤 '결제차단 ..., link: https://news.mt.co.kr/mtview.php?no=2025091012594297684;
snippet: 4 days ago · 경기 광명과 서울 금천지역 KT 이용자들이 동시다발적으로 소액결제 피해를 본 사건과 관련해 경찰이 뚜렷한 단서를 찾지 못하고 있습니다. 서울 ..., title: 'KT 소액결제 사건' 미스터리...경찰 "전례 없는 사건" - YTN, link: https://www.ytn.co.kr/_ln/0103_202509082239409382


In [53]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = question_answering_prompt | llm

In [54]:
from langchain.memory import ChatMessageHistory

# 채팅 메시지를 저장할 메모리 객체 생성
chat_history = ChatMessageHistory() 
# 사용자 질문을 메모리에 저장
chat_history.add_user_message(query) 

# 문서 검색하고 답변 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs, # 검색 결과 전달
    }
)

# 생성된 답변을 메모리에 저장
chat_history.add_ai_message(answer) 

print(answer)

content='KT 소액결제 사건에 대한 정보는 다음과 같습니다.\n\n*   최근 수도권 특정 지역의 KT 이용자들이 휴대전화 소액결제 피해를 보는 사례가 잇따라 발생하고 있습니다.\n*   경기 광명과 서울 금천지역에서 KT 이용자들이 동시다발적으로 소액결제 피해를 본 사건과 관련해 경찰이 수사 중이나 뚜렷한 단서를 찾지 못하고 있습니다.\n*   서울 영등포구에서도 유사한 사건이 발생하여 지난 8월 6일 30대 KT 가입자가 소액결제 피해를 신고했습니다.\n*   KT가 이번 사건 초기 경찰로부터 발생 사실을 통보받았으나, "이상 없음"이라고 답하여 늑장 대처 의혹이 제기되고 있습니다.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []} id='run--2b52d947-e724-426f-86f0-a553fb30ec38-0' usage_metadata={'input_tokens': 565, 'output_tokens': 207, 'total_tokens': 772, 'input_token_details': {'cache_read': 0}}


### 🔎 웹 검색 도구

In [55]:
web_search = DuckDuckGoSearchResults(output_format="list", name="web_search", description="웹 검색이 필요할 때 사용합니다.")

search_results = web_search.invoke(query)

print(search_results)

[{'snippet': '4 days ago · 서울 영등포구 에서도 유사한 사건이 발생했던 것으로 확인됐다. 서울 영등포경찰서는 지난달 (8월) 6일 오후 "소액결제로 49만 5천 원이 빠져나갔다"는 30대 KT 가입자의 신고를 접수했다. 피해자는 "휴대전화 인증 문자를 받은 …', 'title': 'KT 소액결제 해킹 사건 - 나무위키', 'link': 'https://namu.wiki/w/KT+소액결제+해킹+사건'}, {'snippet': '2 days ago · 수도권 특정 지역의 KT 이용자들이 휴대전화 소액결제 피해를 보는 사례가 잇따르는 가운데 KT가 이번 사건 초기 경찰로부터 발생 사실을 통보 ...', 'title': "KT, 소액결제 사건 초기 경찰서 통보받아…'늑장 대처' 의혹", 'link': 'https://news.sbs.co.kr/news/endPage.do?news_id=N1008250962'}, {'snippet': '1 day ago · 지난달 하순부터 확인된 KT의 소액결제 사태와 관련해 경찰이 그 전에 이상징후를 통보했음에도 이를 KT 측이 "이상없음"이라고 답했다는 지적이 제기됐다. 10일 국회 과학기술정보방송통신위원회 소속 황정아 의원실은 "KT는 언론 …', 'title': "KT, '소액결제 사건' 경찰 통보 받고도 무시…4일 뒤 '결제차단 ...", 'link': 'https://news.mt.co.kr/mtview.php?no=2025091012594297684'}, {'snippet': '3 days ago · 경기 광명과 서울 금천지역 KT 이용자들이 동시다발적으로 소액결제 피해를 본 사건과 관련해 경찰이 뚜렷한 단서를 찾지 못하고 있습니다. 서울 ...', 'title': '\'KT 소액결제 사건\' 미스터리...경찰 "전례 없는 사건" - YTN', 'link': 'https://www.ytn.co.kr/_ln/0103_202509082239409382'}]


c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch07\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


In [56]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# 도구들을 tools 리스트에 추가
tools = [web_search] # 에이전트가 사용할 도구 리스트

# 에이전트 프롬프트 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    ("human", "{input}"),
    # AgentExecutor는 agent_scratchpad를 자동으로 채워 넣어,
    # 에이전트가 이전 단계의 행동과 결과를 기억하고 다음 행동을 결정할 수 있도록 함
    ("placeholder", "{agent_scratchpad}"),
])

# 에이전트 생성
agent = create_tool_calling_agent(llm, tools, prompt)

# AgentExecutor 초기화
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [57]:
from langchain_core.messages import HumanMessage

# 메시지 형식 변경 (agent_executor에 맞게)
messages = [
    HumanMessage(query),
]

# agent_executor를 사용하여 실행
response = agent_executor.invoke({"input": messages})
response



> Entering new AgentExecutor chain...

Invoking: `web_search` with `{'query': 'KT 소액결제 사건'}`


[{'snippet': '4 days ago · 서울 영등포구 에서도 유사한 사건이 발생했던 것으로 확인됐다. 서울 영등포경찰서는 지난달 (8월) 6일 오후 "소액결제로 49만 5천 원이 빠져나갔다"는 30대 KT 가입자의 신고를 접수했다. 피해자는 "휴대전화 인증 문자를 받은 적이 없는데 소액결제가 이뤄졌다.', 'title': 'KT 소액결제 해킹 사건 - 나무위키', 'link': 'https://namu.wiki/w/KT+소액결제+해킹+사건'}, {'snippet': '2 days ago · 수도권 특정 지역의 KT 이용자들이 휴대전화 소액결제 피해를 보는 사례가 잇따르는 가운데 KT가 이번 사건 초기 경찰로부터 발생 사실을 통보 ...', 'title': "KT, 소액결제 사건 초기 경찰서 통보받아…'늑장 대처' 의혹", 'link': 'https://news.sbs.co.kr/news/endPage.do?news_id=N1008250962'}, {'snippet': '1 day ago · 지난달 하순부터 확인된 KT의 소액결제 사태와 관련해 경찰이 그 전에 이상징후를 통보했음에도 이를 KT 측이 "이상없음"이라고 답했다는 지적이 제기됐다. 10일 국회 과학기술정보방송통신위원회 소속 황정아 의원실은 "KT는 언론 보도와 경찰 조사 등으로 소액결제 해킹 사고를 인지했음에도 정작 ...', 'title': "KT, '소액결제 사건' 경찰 통보 받고도 무시…4일 뒤 '결제차단 ...", 'link': 'https://news.mt.co.kr/mtview.php?no=2025091012594297684'}, {'snippet': '3 days ago · 경기 광명과 서울 금천지역 KT 이용자들이 동시다발적으로 소액결제 피해를 본 사건과 관련해 경찰이 뚜렷한 단서를 찾지 못하고 있습니다. 

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch07\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


 KT 소액결제 사건에 대한 최신 정보는 다음과 같습니다.

* **최근 발생:** 최근 수도권 특정 지역의 KT 이용자들이 휴대전화 소액결제 피해를 보는 사례가 잇따라 발생하고 있습니다.
* **KT의 대응:** KT는 이번 사건 초기 경찰로부터 발생 사실을 통보받았음에도 늑장 대처 의혹을 받고 있습니다.
* **경찰 수사:** 경찰은 경기 광명과 서울 금천지역 KT 이용자들이 동시다발적으로 소액결제 피해를 본 사건에 대해 수사 중이지만 뚜렷한 단서를 찾지 못하고 있습니다.
* **과거 사례:** 서울 영등포구에서도 유사한 사건이 발생했던 것으로 확인되었습니다.

자세한 내용은 위 링크들을 참고하시면 됩니다.

> Finished chain.


{'input': [HumanMessage(content='KT 소액결제 사건', additional_kwargs={}, response_metadata={})],
 'output': ' KT 소액결제 사건에 대한 최신 정보는 다음과 같습니다.\n\n* **최근 발생:** 최근 수도권 특정 지역의 KT 이용자들이 휴대전화 소액결제 피해를 보는 사례가 잇따라 발생하고 있습니다.\n* **KT의 대응:** KT는 이번 사건 초기 경찰로부터 발생 사실을 통보받았음에도 늑장 대처 의혹을 받고 있습니다.\n* **경찰 수사:** 경찰은 경기 광명과 서울 금천지역 KT 이용자들이 동시다발적으로 소액결제 피해를 본 사건에 대해 수사 중이지만 뚜렷한 단서를 찾지 못하고 있습니다.\n* **과거 사례:** 서울 영등포구에서도 유사한 사건이 발생했던 것으로 확인되었습니다.\n\n자세한 내용은 위 링크들을 참고하시면 됩니다.'}